In [4]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from datasets import load_dataset
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

In [5]:
# Load tokenizer and model
model_name = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = TFAutoModelForSequenceClassification.from_pretrained(model_name)

# Load dataset
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")


All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

Some layers of TFRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

In [6]:
# Function to encode the texts
def encode_data(tokenizer, texts, max_length=512):
    return tokenizer(texts, padding="max_length", truncation=True, max_length=max_length, return_tensors="tf")

In [7]:
# Preprocess the data
def preprocess_data(dataset, split):
    texts = [example['text'] for example in dataset[split]]
    labels = [example['label'] for example in dataset[split]]
    encodings = encode_data(tokenizer, texts)
    dataset = tf.data.Dataset.from_tensor_slices((dict(encodings), labels))
    return dataset.shuffle(1000).batch(8)

train_dataset = preprocess_data(dataset, 'train')
val_dataset = preprocess_data(dataset, 'validation')

In [16]:
# Model architecture
input_ids = Input(shape=(128,), name="input_ids")
attention_mask = Input(shape=(128,), name="attention_mask")
inputs = {'input_ids': input_ids, 'attention_mask': attention_mask}


In [17]:
# BERT embeddings
bert_outputs = bert_model(inputs)[1]
x = GlobalAveragePooling1D()(bert_outputs)
x = Dense(128, activation='relu')(x)
x = Dropout(0.1)(x)
outputs = Dense(1, activation='sigmoid')(x)

ValueError: Exception encountered when calling layer 'tf_roberta_for_sequence_classification' (type TFRobertaForSequenceClassification).

Data of type <class 'keras.src.backend.common.keras_tensor.KerasTensor'> is not allowed only (<class 'tensorflow.python.framework.tensor.Tensor'>, <class 'bool'>, <class 'int'>, <class 'transformers.utils.generic.ModelOutput'>, <class 'tuple'>, <class 'list'>, <class 'dict'>, <class 'numpy.ndarray'>) is accepted for input_ids.

Call arguments received by layer 'tf_roberta_for_sequence_classification' (type TFRobertaForSequenceClassification):
  • input_ids={'input_ids': '<KerasTensor shape=(None, 128), dtype=float32, sparse=None, name=input_ids>', 'attention_mask': '<KerasTensor shape=(None, 128), dtype=float32, sparse=None, name=attention_mask>'}
  • attention_mask=None
  • token_type_ids=None
  • position_ids=None
  • head_mask=None
  • inputs_embeds=None
  • output_attentions=None
  • output_hidden_states=None
  • return_dict=None
  • labels=None


In [ ]:
# Compile model
model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Training
model.fit(train_dataset, validation_data=val_dataset, epochs=3)

In [ ]:
# Evaluate the model
model.evaluate(val_dataset)